# Task 2 — Explore the Data Like a Security Analyst

**Goal:** before modeling, understand *where the attacks live*. Which protocols carry them? Which services get targeted? Which features separate benign from malicious traffic?

**Why it matters:** a model is only as good as the signal in its features. By exploring first, you (a) sanity-check the data, (b) build intuition for *why* detection is even possible, and (c) can explain your model later — 'it fires on high SYN-error rates because those are scans.' That explainability is gold in security interviews.

**The key tool:** `groupby`. The recurring pattern is `df.groupby("some_category")["is_attack"].mean()` — because the mean of a 0/1 column is the *attack rate* within each group. You learned that trick in Task 1; now you'll use it to hunt.

## Step 0 — Reload the data
This is a fresh notebook, so `df` doesn't exist yet. Re-do the Task 1 load here (good practice — repetition builds fluency). You need: import pandas, define `all_columns`, `read_csv`, and add the `is_attack` column as **0/1 int**.

In [2]:
# TODO: import pandas, build all_columns, read the train file into df,
#       and add df["is_attack"] as a 0/1 integer column.
#   (You can copy the column list from notebook 01 / TASKS.md.)
import pandas as pd
feature_columns=[
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
]

all_columns=feature_columns+["label","difficulty"]

df=pd.read_csv("/Users/rohanb/06_projects/CYBERSECURITY/data/raw/KDDTrain+.txt", header=None, names=all_columns)
df["is_attack"]=df["label"] != "normal"
df

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty,is_attack
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20,False
1,0,udp,other,SF,146,0,0,0,0,0,...,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15,False
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19,True
3,0,tcp,http,SF,232,8153,0,0,0,0,...,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21,False
4,0,tcp,http,SF,199,420,0,0,0,0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.06,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20,True
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.01,0.01,0.00,0.00,0.00,0.00,0.00,normal,21,False
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.06,0.00,0.00,0.72,0.00,0.01,0.00,normal,18,False
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20,True


## Step 1 — The overall picture
What fraction of all connections are attacks? Print the count of each class and the overall attack rate.

**Hints:** `df["is_attack"].value_counts()` for counts; `df["is_attack"].mean()` for the rate.

In [3]:
# TODO: show the class counts and the overall attack rate
df["is_attack"].value_counts()

is_attack
False    67343
True     58630
Name: count, dtype: int64

In [5]:
print(df["is_attack"].mean())

0.4654171925730117


In [6]:
df

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty,is_attack
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20,False
1,0,udp,other,SF,146,0,0,0,0,0,...,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15,False
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19,True
3,0,tcp,http,SF,232,8153,0,0,0,0,...,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21,False
4,0,tcp,http,SF,199,420,0,0,0,0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.06,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20,True
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.01,0.01,0.00,0.00,0.00,0.00,0.00,normal,21,False
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.06,0.00,0.00,0.72,0.00,0.01,0.00,normal,18,False
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20,True


## Step 2 — Which PROTOCOL carries the most attacks?
Network traffic mostly rides on 3 protocols: **tcp** (reliable, connection-based — web, ssh, email), **udp** (fire-and-forget — DNS, streaming), **icmp** (control/ping — often abused for floods).

Compute, for each `protocol_type`: how many connections there are, and what fraction are attacks.

**Hints:** `df.groupby("protocol_type")["is_attack"].agg(["count", "mean"])`. Round with `.round(3)` for readability.

In [9]:
# TODO: attack count and rate per protocol_type
df.groupby("protocol_type")["is_attack"].agg(["count","mean"]).round(3)

,count,mean
protocol_type,,
icmp,8291,0.842
tcp,102689,0.478
udp,14993,0.171


## Step 3 — Which SERVICES get targeted?
A `service` is the app/port the connection is talking to (http=web, smtp=email, ftp_data=file transfer, private, etc.). There are many, so focus on the busiest ones.

Find the **top 10 most common services**, then show the attack count and attack rate for just those.

**Hints:**
- `df["service"].value_counts().head(10).index` gives you the 10 busiest service names.
- Filter with `df[df["service"].isin(those_names)]`, then `groupby("service")["is_attack"].agg(["count", "mean"])`.
- Which service is almost ALL attacks? Which is almost all benign? Note them — that contrast is real signal.

In [16]:
# TODO: top-10 services with their attack count and rate
busy_services=df["service"].value_counts().head(10).index
busy_services

Index(['http', 'private', 'domain_u', 'smtp', 'ftp_data', 'eco_i', 'other',
       'ecr_i', 'telnet', 'finger'],
      dtype='str', name='service')

In [18]:
df[df["service"].isin(busy_services)].groupby("service")["is_attack"].agg(["count","mean"])

,count,mean
service,,
domain_u,9043,0.000995
eco_i,4586,0.891627
ecr_i,3077,0.938252
finger,1767,0.691568
ftp_data,6860,0.273469
http,40338,0.056746
other,4359,0.402615
private,21853,0.955063
smtp,7313,0.038835


## Step 4 — Find a discriminative feature
A *discriminative* feature has very different values for benign vs attack traffic — that's what a model exploits. Let's test one: `serror_rate` (the fraction of a connection's activity that hit SYN errors — the fingerprint of port scans / SYN floods).

Compute the **average `serror_rate` for benign vs attack** connections.

**Hints:** `df.groupby("is_attack")["serror_rate"].mean()`. If the two numbers are far apart, the feature is discriminative.

In [22]:
# TODO: average serror_rate for benign vs attack
average_serror_rate=df.groupby("is_attack")["serror_rate"].mean()
print(average_serror_rate)

is_attack
False    0.013441
True     0.595808
Name: serror_rate, dtype: float64


## Step 5 — Your turn to investigate (no hints)
Pick **one more** numeric feature you're curious about (e.g. `src_bytes`, `count`, `dst_host_srv_count`, `same_srv_rate`) and check whether it's discriminative — compare its average for benign vs attack the same way as Step 4. Write one sentence: is it a good signal or not, and does it make intuitive sense?

In [23]:
# TODO: your own benign-vs-attack comparison for a feature of your choice
df

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty,is_attack
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20,False
1,0,udp,other,SF,146,0,0,0,0,0,...,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15,False
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19,True
3,0,tcp,http,SF,232,8153,0,0,0,0,...,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21,False
4,0,tcp,http,SF,199,420,0,0,0,0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.06,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20,True
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.01,0.01,0.00,0.00,0.00,0.00,0.00,normal,21,False
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.06,0.00,0.00,0.72,0.00,0.01,0.00,normal,18,False
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20,True


In [25]:
average_src_bytes=df.groupby("is_attack")["src_bytes"].mean()
print(average_src_bytes)

is_attack
False    13133.279331
True     82820.141327
Name: src_bytes, dtype: float64


### ✅ You pass Task 2 when you can answer:
1. What's the overall attack rate? (~46.5%)
2. Which protocol has the highest attack rate — and does that match its reputation (icmp abused for floods)?
3. Name one service that's almost all attacks and one that's almost all benign.
4. Is `serror_rate` discriminative? By roughly how many times?
5. The feature YOU picked in Step 5 — discriminative or not, and why?

Paste me your outputs + your one-sentence Step 5 finding, and I'll review + we move to Task 3 (turning categories into numbers the model can use).